# SAM 3.1 tracking on Colab (free T4 GPU)

Role A / Phase 1. Run this with **Runtime > Change runtime type > T4 GPU** selected.

GPU reality check (see `docs/COMPUTE.md`): the hackathon provides no GPUs, RunPod needs a booth
visit, our only physical GPU (a K1900, ~2GB) is below SAM 3.1's ~4GB floor. Colab's free T4
(16GB, real CUDA) clears that easily and costs nothing — that's why we're here.

We use the **official 🤗 Transformers path** (`Sam3VideoModel`/`Sam3VideoProcessor`, straight
from the model card at https://huggingface.co/facebook/sam3) — weights auto-download via
`from_pretrained()` once your HF account has approved access, no manual file placement needed.

**Known risk:** SAM 3 weights are gated and access is **not guaranteed** — people have been
denied. If Step 3 below doesn't clear in time, stop and use the fal.ai path instead (no GPU
needed at all): see `docs/DEFERRED.md`. Or keep developing everything else against
`sam.backend: replay` (already verified, works today, no GPU).

In [ ]:
# 1. Confirm we actually got a GPU runtime (fails loud, exactly like src.utils.gpu)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clone/update the repo and install deps (CUDA torch ships with Colab)
import os

REPO_URL = "https://github.com/wheredawoodat949/AI-Hackathon"
BRANCH = "feat/tracking-ashmeet"

if not os.path.isdir("AI-Hackathon"):
    !git clone --branch {BRANCH} {REPO_URL}.git
%cd AI-Hackathon
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
%pip install -q -r requirements.txt
%pip install -q -U "transformers>=5.12.1" accelerate huggingface_hub
!python -m src.utils.gpu        # must print CUDA — if not, fix the runtime type above

import transformers
print("transformers:", transformers.__version__)
print("If the next cell reports an older in-memory Transformers version, use Runtime > Restart session, then resume at Step 3.")

## 3. SAM 3 access (gated — the one step that can block this whole path)

1. Request access: https://huggingface.co/facebook/sam3 (sign the agreement). **Can be denied
   or take time to approve** — check status any time at https://huggingface.co/settings/gated-repos.
2. Create a read token: https://huggingface.co/settings/tokens
3. Paste it below and log in — `from_pretrained("facebook/sam3")` will then auto-download the
   weights the first time `sam_local.py` needs them. No manual file download/placement needed.

In [ ]:
import getpass
import os

from huggingface_hub import HfApi, login

# huggingface_hub prioritizes HF_TOKEN over login(). Remove any stale token
# inherited from an earlier Colab run before authenticating this session.
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
hf_token = getpass.getpass("HF token (input hidden): ").strip()
login(token=hf_token, add_to_git_credential=False)
identity = HfApi().whoami(token=hf_token)
# Make this exact, newly verified token available to the tracking subprocess.
os.environ["HF_TOKEN"] = hf_token
print("Authenticated Hugging Face account:", identity["name"])

In [ ]:
# Access/API smoke test. Do not continue until this prints Access OK.
# Direct downloads expose the real Hub/auth error that Transformers otherwise wraps
# as the misleading generic message: "Can't load image processor".
from packaging.version import Version
from huggingface_hub import hf_hub_download
import transformers

if Version(transformers.__version__) < Version("5.12.1"):
    raise RuntimeError(
        f"Transformers {transformers.__version__} is still loaded; need >=5.12.1. "
        "Runtime > Restart session, then resume at Step 3."
    )
if "hf_token" not in globals() or not hf_token:
    raise RuntimeError("Run the Step 3 login cell again; no verified token is in this kernel.")

for filename in ("config.json", "processor_config.json"):
    hf_hub_download(
        repo_id="facebook/sam3", filename=filename, token=hf_token, force_download=True
    )
    print("Access verified:", filename)

from transformers import Sam3VideoProcessor
processor = Sam3VideoProcessor.from_pretrained(
    "facebook/sam3", token=hf_token, force_download=True
)
print("Access OK — gated repo and processor load fine.")

In [ ]:
# 4. Locate the real match video, then create a bounded 10-second/1280px clip.
# A drive.google.com URL is a web page, not a mounted filesystem path. This cell
# first reuses data/videos, then refreshes and searches My Drive recursively.
from pathlib import Path
import subprocess

MATCH = "117093"
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19JQ9Uq1doemU8PlGDs5oPlNTpi6L0xnS"

def usable_videos(root):
    root = Path(root)
    if not root.exists():
        return []
    return [p for p in root.rglob(f"{MATCH}*.mp4") if p.is_file() and p.stat().st_size > 1_000_000]

def video_rank(path):
    name = path.name.lower()
    return (
        "calibrat" in name,   # prefer the normal panorama
        "2nd" in name,        # prefer first half
        "panorama" not in name,
        len(name),
    )

candidates = usable_videos("data/videos")
if not candidates:
    from google.colab import drive
    # The dataset folder was moved into My Drive after the earlier mount. Force a
    # refresh so Colab sees that move instead of searching a stale mount.
    print("Refreshing Google Drive mount...")
    drive.mount("/content/drive", force_remount=True)
    print("Searching /content/drive/MyDrive for the match video...")
    candidates = usable_videos("/content/drive/MyDrive")
    candidates += usable_videos("/content/drive/Shareddrives")
    # Colab exposes Drive shortcuts under this hidden directory.
    candidates += usable_videos("/content/drive/.shortcut-targets-by-id")

if not candidates:
    # The user-provided folder is link-accessible but is not automatically mounted.
    # List it with gdown and download only the preferred normal first-half panorama.
    import gdown
    print("No mounted copy found; listing the shared folder directly...")
    entries = gdown.download_folder(
        url=GDRIVE_FOLDER_URL, skip_download=True, quiet=False, use_cookies=False
    ) or []
    video_entries = [
        entry for entry in entries
        if Path(entry.path).name.startswith(MATCH) and Path(entry.path).suffix.lower() == ".mp4"
    ]
    if not video_entries:
        raise FileNotFoundError(f"No {MATCH}*.mp4 files were listed in {GDRIVE_FOLDER_URL}")
    selected_entry = sorted(video_entries, key=lambda entry: video_rank(Path(entry.path)))[0]
    destination = Path("data/videos") / Path(selected_entry.path).name
    destination.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading selected Drive file:", selected_entry.path)
    downloaded = gdown.download(
        id=selected_entry.id, output=str(destination), quiet=False, use_cookies=False, resume=True
    )
    if not downloaded or not destination.exists() or destination.stat().st_size <= 1_000_000:
        raise RuntimeError("Google Drive did not return a usable video file.")
    candidates = [destination]

candidates = sorted(set(candidates), key=video_rank)
print("video candidates:")
for candidate in candidates:
    print(" -", candidate)
source_video = str(candidates[0])
print("selected source:", source_video)

# Fail here with a useful codec/container error if the selected file is not a video.
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", source_video,
], check=True, capture_output=True, text=True)
print("source duration (seconds):", probe.stdout.strip())

Path("data/clips").mkdir(parents=True, exist_ok=True)
video = "data/clips/117093_10s_1280.mp4"
subprocess.run([
    "ffmpeg", "-y", "-ss", "0", "-i", source_video, "-t", "10",
    "-vf", "scale=1280:-2", "-an", "-c:v", "libx264", "-preset", "veryfast", video,
], check=True)
print("clip ready:", video, Path(video).stat().st_size, "bytes")

In [ ]:
# 5. Run real SAM tracking. Stream the complete console to the notebook and a log file.
import os
import subprocess
import sys

os.makedirs("outputs", exist_ok=True)
console_log = "outputs/sam_local_console.txt"
cmd = [sys.executable, "-m", "src.tracking.demo", "--backend", "local", "--video", video, "--seconds", "10"]
print("tracking:", " ".join(cmd), flush=True)
with open(console_log, "w") as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
print(f"[tracking] exit_code={return_code}; full console: {console_log}")
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)

In [ ]:
# 6. Sanity-check + preview the output (download it from the Colab file browser too).
import glob

out = sorted(glob.glob("outputs/track_117093_local*"))
print(out)

from IPython.display import Video
Video(out[0], embed=True, width=640) if out else None

## If this worked
Set `sam.backend: local` in `config.yaml` on the repo (or just keep using `--backend local`
from here), commit `outputs/track_117093_local.mp4` somewhere visible (it's gitignored —
download it locally / drop it in the team Drive), and tell Role B (pitch+eval) real tracked
detections are available for the homography + HOTA work.

## If Step 3 (access) never cleared
Don't burn more time waiting. Two real fallbacks, both already implemented:
- `docs/DEFERRED.md` → fal.ai hosted SAM 3.1, no GPU/gating, ~$0.16/clip.
- `sam.backend: replay` → already verified, runs the whole pipeline on GSR ground truth, no GPU.